# Brain Tumor MRI Classification: Leakage-Safe Compression–Performance Study

This notebook implements the improved leakage-safe experimental strategy:

1. Combine the original `Training` and `Testing` folders.
2. Split original images into stratified train/validation/test subsets.
3. Apply conservative zoom-in augmentation **only to the training subset**.
4. Do not augment validation or test sets.
5. Train a CNN baseline.
6. Train a dense EfficientNet-B0 using two-phase fine-tuning.
7. Apply L1-norm structured pruning to EfficientNet-B0 at 20%, 40%, 60%, and 80% sparsity.
8. Evaluate accuracy, macro F1-score, model size, compressed size, and inference latency.

This version avoids the main leakage risk from augmenting before splitting, where an original image and its augmented copy could land in different splits.

## 1. Imports and configuration

In [1]:
import os
import random
import copy
import time
import gzip
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
import torchvision.models as models

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Reproducibility
SEED = 101
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# Dataset and experiment settings
DATA_DIR = "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri"
OUTPUT_DIR = "/kaggle/working"

LABELS = [
    "glioma_tumor",
    "no_tumor",
    "meningioma_tumor",
    "pituitary_tumor"
]

label_to_idx = {label: idx for idx, label in enumerate(LABELS)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Dataset folder exists:", os.path.exists(DATA_DIR))


Device: cuda
Dataset folder exists: True


## 2. Data preparation: combine folders without augmentation

In [2]:
def zoom_in_image(image, zoom_range=(0.85, 0.95), output_size=IMAGE_SIZE):
    """Create a conservative zoom-in copy using a random crop and resize.

    Medical-image safety choice: no flip, no rotation, no brightness or contrast change.
    This function is used only after splitting, and only for training images.
    """
    width, height = image.size
    crop_scale = random.uniform(*zoom_range)

    crop_width = int(width * crop_scale)
    crop_height = int(height * crop_scale)

    max_left = width - crop_width
    max_top = height - crop_height

    left = random.randint(0, max_left) if max_left > 0 else 0
    top = random.randint(0, max_top) if max_top > 0 else 0
    right = left + crop_width
    bottom = top + crop_height

    return image.crop((left, top, right, bottom)).resize(
        (output_size, output_size),
        Image.BILINEAR
    )


def load_combined_original_dataset():
    """Load original images from both Training and Testing folders without augmentation."""
    images = []
    targets = []

    for split in ["Training", "Testing"]:
        for label in LABELS:
            folder_path = os.path.join(DATA_DIR, split, label)

            for img_name in tqdm(os.listdir(folder_path), desc=f"{split}/{label}"):
                img_path = os.path.join(folder_path, img_name)
                image = Image.open(img_path).convert("RGB")
                image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)

                images.append(np.array(image))
                targets.append(label_to_idx[label])

    return np.array(images), np.array(targets)


X, y = load_combined_original_dataset()
X, y = shuffle(X, y, random_state=SEED)

print("\nCombined original dataset")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
for label in LABELS:
    print(label, np.sum(y == label_to_idx[label]))


Testing/pituitary_tumor: 100%|██████████| 74/74 [00:00<00:00, 81.43it/s]



Combined original dataset
X shape: (3264, 224, 224, 3)
y shape: (3264,)

Class distribution:
glioma_tumor 926
no_tumor 500
meningioma_tumor 937
pituitary_tumor 901


## 3. Stratified split, training-only zoom augmentation, and DataLoaders

In [3]:
# 80/10/10 stratified split on original images only
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Dataset split before augmentation:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


def augment_training_set_with_zoom(images, labels, augment_per_image=1):
    """Add conservative zoom-in copies to the training set only.

    Validation and test sets remain untouched to avoid leakage and preserve evaluation integrity.
    """
    augmented_images = []
    augmented_labels = []

    for image_array, label in tqdm(list(zip(images, labels)), desc="Zoom augmenting training set"):
        image = Image.fromarray(image_array)

        # Keep original training image
        augmented_images.append(np.array(image))
        augmented_labels.append(int(label))

        # Add zoom-in copy/copies only to training data
        for _ in range(augment_per_image):
            aug_image = zoom_in_image(image)
            augmented_images.append(np.array(aug_image))
            augmented_labels.append(int(label))

    return np.array(augmented_images), np.array(augmented_labels)


X_train, y_train = augment_training_set_with_zoom(
    X_train,
    y_train,
    augment_per_image=1
)

X_train, y_train = shuffle(X_train, y_train, random_state=SEED)

print("\nDataset split after training-only augmentation:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTrain class distribution after augmentation:")
for label in LABELS:
    print(label, np.sum(y_train == label_to_idx[label]))

print("\nValidation class distribution:")
for label in LABELS:
    print(label, np.sum(y_val == label_to_idx[label]))

print("\nTest class distribution:")
for label in LABELS:
    print(label, np.sum(y_test == label_to_idx[label]))


class BrainTumorArrayDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx])
        label = int(self.labels[idx])

        if self.transform:
            image = self.transform(image)

        return image, label


# No extra online augmentation here because zoom copies were already added offline to training only.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = BrainTumorArrayDataset(X_train, y_train, transform=transform)
val_dataset = BrainTumorArrayDataset(X_val, y_val, transform=transform)
test_dataset = BrainTumorArrayDataset(X_test, y_test, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("\nDataLoaders ready.")


Dataset split before augmentation:
Train: (2611, 224, 224, 3)
Validation: (326, 224, 224, 3)
Test: (327, 224, 224, 3)


Zoom augmenting training set: 100%|██████████| 2611/2611 [00:03<00:00, 861.70it/s]



Dataset split after training-only augmentation:
Train: (5222, 224, 224, 3)
Validation: (326, 224, 224, 3)
Test: (327, 224, 224, 3)

Train class distribution after augmentation:
glioma_tumor 1482
no_tumor 800
meningioma_tumor 1498
pituitary_tumor 1442

Validation class distribution:
glioma_tumor 92
no_tumor 50
meningioma_tumor 94
pituitary_tumor 90

Test class distribution:
glioma_tumor 93
no_tumor 50
meningioma_tumor 94
pituitary_tumor 90

DataLoaders ready.


## 4. Shared training, evaluation, and reporting utilities

In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels_batch in loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels_batch in loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels_batch)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    loss = running_loss / total
    acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return loss, acc, macro_f1, all_labels, all_preds


def print_test_results(model_name, y_true, y_pred, test_loss, test_acc, test_f1):
    print(f"\n{model_name} Test Results")
    print("-" * 45)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=4, zero_division=0))


def save_results_csv(rows, filename):
    df = pd.DataFrame(rows)
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    display(df)
    return df


## 5. Model 1: CNN baseline

In [5]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((7, 7))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## 6. Train and evaluate CNN baseline

In [6]:
cnn = CNNBaseline(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-4)

cnn_epochs = 50
best_val_f1 = 0.0
best_cnn_wts = copy.deepcopy(cnn.state_dict())
cnn_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(cnn_epochs):
    train_loss, train_acc = train_one_epoch(cnn, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, val_f1, _, _ = evaluate(cnn, val_loader, criterion, DEVICE)

    cnn_history["train_loss"].append(train_loss)
    cnn_history["train_acc"].append(train_acc)
    cnn_history["val_loss"].append(val_loss)
    cnn_history["val_acc"].append(val_acc)
    cnn_history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_cnn_wts = copy.deepcopy(cnn.state_dict())
        torch.save(cnn.state_dict(), os.path.join(OUTPUT_DIR, "best_cnn_baseline.pth"))

    print(f"Epoch [{epoch+1}/{cnn_epochs}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

cnn.load_state_dict(best_cnn_wts)
cnn_test_loss, cnn_test_acc, cnn_test_f1, y_true_cnn, y_pred_cnn = evaluate(cnn, test_loader, criterion, DEVICE)
print_test_results("CNN Baseline", y_true_cnn, y_pred_cnn, cnn_test_loss, cnn_test_acc, cnn_test_f1)

cnn_results = save_results_csv([{
    "model": "CNN Baseline",
    "test_accuracy": cnn_test_acc,
    "test_macro_f1": cnn_test_f1,
    "test_loss": cnn_test_loss,
    "epochs": cnn_epochs
}], "cnn_baseline_results.csv")


Epoch [1/50] Train Acc: 0.5833 | Val Acc: 0.6810 | Val F1: 0.6626
Epoch [2/50] Train Acc: 0.6948 | Val Acc: 0.7577 | Val F1: 0.7536
Epoch [3/50] Train Acc: 0.7411 | Val Acc: 0.7730 | Val F1: 0.7729
Epoch [4/50] Train Acc: 0.7748 | Val Acc: 0.7761 | Val F1: 0.7647
Epoch [5/50] Train Acc: 0.7827 | Val Acc: 0.8190 | Val F1: 0.8173
Epoch [6/50] Train Acc: 0.8070 | Val Acc: 0.8282 | Val F1: 0.8281
Epoch [7/50] Train Acc: 0.8340 | Val Acc: 0.7975 | Val F1: 0.7986
Epoch [8/50] Train Acc: 0.8405 | Val Acc: 0.8190 | Val F1: 0.8201
Epoch [9/50] Train Acc: 0.8570 | Val Acc: 0.8497 | Val F1: 0.8501
Epoch [10/50] Train Acc: 0.8773 | Val Acc: 0.8344 | Val F1: 0.8342
Epoch [11/50] Train Acc: 0.8797 | Val Acc: 0.8834 | Val F1: 0.8863
Epoch [12/50] Train Acc: 0.8916 | Val Acc: 0.8650 | Val F1: 0.8667
Epoch [13/50] Train Acc: 0.8964 | Val Acc: 0.8497 | Val F1: 0.8537
Epoch [14/50] Train Acc: 0.9064 | Val Acc: 0.8742 | Val F1: 0.8824
Epoch [15/50] Train Acc: 0.9056 | Val Acc: 0.8988 | Val F1: 0.9042
Epoc

,model,test_accuracy,test_macro_f1,test_loss,epochs
0,CNN Baseline,0.954128,0.952665,0.164289,50


## 7. Model 2: EfficientNet-B0 dense model

In [7]:
def create_efficientnet_b0(num_classes=4):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, num_classes)
    )
    return model


## 8. Train and evaluate Dense EfficientNet-B0

In [8]:
effnet = create_efficientnet_b0(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()

# Phase 1: train classifier head only
for param in effnet.features.parameters():
    param.requires_grad = False
for param in effnet.classifier.parameters():
    param.requires_grad = True

phase1_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, effnet.parameters()),
    lr=1e-3,
    weight_decay=1e-2
)

phase1_epochs = 20
print("\nPhase 1: Training classifier head only")
print("-" * 45)

for epoch in range(phase1_epochs):
    train_loss, train_acc = train_one_epoch(effnet, train_loader, criterion, phase1_optimizer, DEVICE)
    val_loss, val_acc, val_f1, _, _ = evaluate(effnet, val_loader, criterion, DEVICE)
    print(f"Phase 1 Epoch [{epoch+1}/{phase1_epochs}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

# Phase 2: fine-tune later EfficientNet blocks
for param in effnet.features.parameters():
    param.requires_grad = False

feature_children = list(effnet.features.children())
for child in feature_children[-3:]:
    for param in child.parameters():
        param.requires_grad = True

for param in effnet.classifier.parameters():
    param.requires_grad = True

phase2_epochs = 50
phase2_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, effnet.parameters()),
    lr=1e-4,
    weight_decay=1e-2
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(phase2_optimizer, T_max=phase2_epochs)

best_val_f1 = 0.0
best_effnet_wts = copy.deepcopy(effnet.state_dict())
effnet_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

print("\nPhase 2: Fine-tuning top EfficientNet layers")
print("-" * 45)

for epoch in range(phase2_epochs):
    train_loss, train_acc = train_one_epoch(effnet, train_loader, criterion, phase2_optimizer, DEVICE)
    val_loss, val_acc, val_f1, _, _ = evaluate(effnet, val_loader, criterion, DEVICE)
    scheduler.step()

    effnet_history["train_loss"].append(train_loss)
    effnet_history["train_acc"].append(train_acc)
    effnet_history["val_loss"].append(val_loss)
    effnet_history["val_acc"].append(val_acc)
    effnet_history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_effnet_wts = copy.deepcopy(effnet.state_dict())
        torch.save(effnet.state_dict(), os.path.join(OUTPUT_DIR, "best_efficientnet_b0.pth"))

    print(f"Phase 2 Epoch [{epoch+1}/{phase2_epochs}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

effnet.load_state_dict(best_effnet_wts)
effnet_test_loss, effnet_test_acc, effnet_test_f1, y_true_effnet, y_pred_effnet = evaluate(effnet, test_loader, criterion, DEVICE)
print_test_results("EfficientNet-B0 Dense", y_true_effnet, y_pred_effnet, effnet_test_loss, effnet_test_acc, effnet_test_f1)

effnet_results = save_results_csv([{
    "model": "EfficientNet-B0 Dense",
    "test_accuracy": effnet_test_acc,
    "test_macro_f1": effnet_test_f1,
    "test_loss": effnet_test_loss,
    "phase1_epochs": phase1_epochs,
    "phase2_epochs": phase2_epochs
}], "efficientnet_b0_dense_results.csv")


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 141MB/s]



Phase 1: Training classifier head only
---------------------------------------------
Phase 1 Epoch [1/20] Train Acc: 0.7074 | Val Acc: 0.8006 | Val F1: 0.8012
Phase 1 Epoch [2/20] Train Acc: 0.7777 | Val Acc: 0.8160 | Val F1: 0.8143
Phase 1 Epoch [3/20] Train Acc: 0.7995 | Val Acc: 0.8374 | Val F1: 0.8381
Phase 1 Epoch [4/20] Train Acc: 0.7991 | Val Acc: 0.8405 | Val F1: 0.8443
Phase 1 Epoch [5/20] Train Acc: 0.8164 | Val Acc: 0.8313 | Val F1: 0.8337
Phase 1 Epoch [6/20] Train Acc: 0.8198 | Val Acc: 0.8344 | Val F1: 0.8403
Phase 1 Epoch [7/20] Train Acc: 0.8175 | Val Acc: 0.8497 | Val F1: 0.8508
Phase 1 Epoch [8/20] Train Acc: 0.8219 | Val Acc: 0.8558 | Val F1: 0.8618
Phase 1 Epoch [9/20] Train Acc: 0.8263 | Val Acc: 0.8620 | Val F1: 0.8668
Phase 1 Epoch [10/20] Train Acc: 0.8238 | Val Acc: 0.8558 | Val F1: 0.8595
Phase 1 Epoch [11/20] Train Acc: 0.8219 | Val Acc: 0.8558 | Val F1: 0.8609
Phase 1 Epoch [12/20] Train Acc: 0.8261 | Val Acc: 0.8466 | Val F1: 0.8486
Phase 1 Epoch [13/20] T

KeyboardInterrupt: 

## 9. Model 3 utilities: structured pruning, size, and latency

In [ ]:
def apply_structured_pruning(model, amount):
    """Apply L1-norm structured pruning to every Conv2d layer."""
    pruned_layers = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(module, name="weight", amount=amount, n=1, dim=0)
            pruned_layers.append(name)

    print(f"Pruned {len(pruned_layers)} Conv2d layers at {int(amount * 100)}%")
    return model, pruned_layers


def remove_pruning_reparameterization(model):
    """Make pruning permanent after fine-tuning."""
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            try:
                prune.remove(module, "weight")
            except ValueError:
                pass
    return model


def calculate_total_sparsity(model):
    total_params = 0
    zero_params = 0
    for param in model.parameters():
        total_params += param.numel()
        zero_params += torch.sum(param == 0).item()
    return zero_params / total_params if total_params else 0, zero_params, total_params


def calculate_conv_sparsity(model):
    total_params = 0
    zero_params = 0
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            weight = module.weight
            total_params += weight.numel()
            zero_params += torch.sum(weight == 0).item()
    return zero_params / total_params if total_params else 0, zero_params, total_params


def get_model_size_mb(model, filename):
    torch.save(model.state_dict(), filename)
    return os.path.getsize(filename) / (1024 * 1024)


def get_compressed_model_size_mb(model, filename):
    with gzip.open(filename, "wb") as f:
        pickle.dump(model.state_dict(), f)
    return os.path.getsize(filename) / (1024 * 1024)


def measure_inference_time(model, loader, device, num_batches=20):
    model.eval()
    times = []
    total_images = 0

    # Warm-up
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):
            if batch_idx >= 3:
                break
            _ = model(images.to(device))

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Timed inference
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):
            if batch_idx >= num_batches:
                break

            images = images.to(device)
            if device.type == "cuda":
                torch.cuda.synchronize()

            start_time = time.time()
            _ = model(images)

            if device.type == "cuda":
                torch.cuda.synchronize()

            times.append(time.time() - start_time)
            total_images += images.size(0)

    return (sum(times) / total_images) * 1000


## 10. Evaluate dense model size and latency before pruning

In [ ]:
criterion = nn.CrossEntropyLoss()
dense_model_path = os.path.join(OUTPUT_DIR, "best_efficientnet_b0.pth")

dense_model = create_efficientnet_b0(num_classes=NUM_CLASSES)
dense_model.load_state_dict(torch.load(dense_model_path, map_location=DEVICE))
dense_model = dense_model.to(DEVICE)

dense_test_loss, dense_test_acc, dense_test_f1, _, _ = evaluate(dense_model, test_loader, criterion, DEVICE)
dense_total_sparsity, dense_zero_params, dense_total_params = calculate_total_sparsity(dense_model)
dense_conv_sparsity, dense_conv_zero_params, dense_conv_total_params = calculate_conv_sparsity(dense_model)

dense_model_size = get_model_size_mb(dense_model, os.path.join(OUTPUT_DIR, "dense_efficientnet_b0.pth"))
dense_compressed_size = get_compressed_model_size_mb(dense_model, os.path.join(OUTPUT_DIR, "dense_efficientnet_b0.pkl.gz"))
dense_inference_time = measure_inference_time(dense_model, test_loader, DEVICE)

print("\nDense EfficientNet-B0 Baseline")
print("-" * 45)
print(f"Test Accuracy: {dense_test_acc:.4f}")
print(f"Test Macro F1: {dense_test_f1:.4f}")
print(f"Total Sparsity: {dense_total_sparsity:.4f}")
print(f"Conv Sparsity: {dense_conv_sparsity:.4f}")
print(f"Model Size: {dense_model_size:.2f} MB")
print(f"Compressed Size: {dense_compressed_size:.2f} MB")
print(f"Inference Time: {dense_inference_time:.4f} ms/image")


## 11. Run structured pruning experiments

In [ ]:
pruning_levels = [0.20, 0.40, 0.60, 0.80]
fine_tune_epochs = 35

pruning_results = [{
    "model": "EfficientNet-B0 Dense",
    "target_pruning": 0.00,
    "total_sparsity": dense_total_sparsity,
    "conv_sparsity": dense_conv_sparsity,
    "test_accuracy": dense_test_acc,
    "test_macro_f1": dense_test_f1,
    "test_loss": dense_test_loss,
    "model_size_mb": dense_model_size,
    "compressed_size_mb": dense_compressed_size,
    "inference_time_ms_per_image": dense_inference_time,
    "zero_params": dense_zero_params,
    "total_params": dense_total_params,
    "conv_zero_params": dense_conv_zero_params,
    "conv_total_params": dense_conv_total_params
}]

for amount in pruning_levels:
    print("\n" + "=" * 70)
    print(f"Structured pruning EfficientNet-B0 at {int(amount * 100)}%")
    print("=" * 70)

    pruned_model = create_efficientnet_b0(num_classes=NUM_CLASSES)
    pruned_model.load_state_dict(torch.load(dense_model_path, map_location=DEVICE))
    pruned_model = pruned_model.to(DEVICE)

    pruned_model, _ = apply_structured_pruning(pruned_model, amount=amount)

    initial_total_sparsity, _, _ = calculate_total_sparsity(pruned_model)
    initial_conv_sparsity, _, _ = calculate_conv_sparsity(pruned_model)
    print(f"Initial Total Sparsity: {initial_total_sparsity:.4f}")
    print(f"Initial Conv Sparsity: {initial_conv_sparsity:.4f}")

    optimizer = optim.AdamW(pruned_model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=fine_tune_epochs)

    best_val_f1 = 0.0
    best_model_wts = copy.deepcopy(pruned_model.state_dict())

    for epoch in range(fine_tune_epochs):
        train_loss, train_acc = train_one_epoch(pruned_model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_f1, _, _ = evaluate(pruned_model, val_loader, criterion, DEVICE)
        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(pruned_model.state_dict())

        print(
            f"Epoch [{epoch+1}/{fine_tune_epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
        )

    pruned_model.load_state_dict(best_model_wts)
    pruned_model = remove_pruning_reparameterization(pruned_model)

    final_total_sparsity, zero_params, total_params = calculate_total_sparsity(pruned_model)
    final_conv_sparsity, conv_zero_params, conv_total_params = calculate_conv_sparsity(pruned_model)

    test_loss, test_acc, test_f1, y_true_pruned, y_pred_pruned = evaluate(pruned_model, test_loader, criterion, DEVICE)

    model_filename = os.path.join(OUTPUT_DIR, f"efficientnet_b0_pruned_{int(amount * 100)}.pth")
    compressed_filename = os.path.join(OUTPUT_DIR, f"efficientnet_b0_pruned_{int(amount * 100)}.pkl.gz")

    model_size = get_model_size_mb(pruned_model, model_filename)
    compressed_size = get_compressed_model_size_mb(pruned_model, compressed_filename)
    inference_time = measure_inference_time(pruned_model, test_loader, DEVICE)

    print("\nPruned Model Test Results")
    print("-" * 45)
    print(f"Target Pruning: {int(amount * 100)}%")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")
    print(f"Total Sparsity: {final_total_sparsity:.4f}")
    print(f"Conv Sparsity: {final_conv_sparsity:.4f}")
    print(f"Model Size: {model_size:.2f} MB")
    print(f"Compressed Size: {compressed_size:.2f} MB")
    print(f"Inference Time: {inference_time:.4f} ms/image")
    print("\nClassification Report:")
    print(classification_report(y_true_pruned, y_pred_pruned, target_names=LABELS, digits=4, zero_division=0))

    pruning_results.append({
        "model": f"EfficientNet-B0 Pruned {int(amount * 100)}%",
        "target_pruning": amount,
        "total_sparsity": final_total_sparsity,
        "conv_sparsity": final_conv_sparsity,
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
        "test_loss": test_loss,
        "model_size_mb": model_size,
        "compressed_size_mb": compressed_size,
        "inference_time_ms_per_image": inference_time,
        "zero_params": zero_params,
        "total_params": total_params,
        "conv_zero_params": conv_zero_params,
        "conv_total_params": conv_total_params
    })

pruning_df = save_results_csv(pruning_results, "efficientnet_b0_improved_pruning_results.csv")


## 12. Visualise compression–performance trade-off

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(pruning_df["conv_sparsity"] * 100, pruning_df["test_accuracy"] * 100, marker="o", label="Accuracy")
plt.plot(pruning_df["conv_sparsity"] * 100, pruning_df["test_macro_f1"] * 100, marker="o", label="Macro F1")
plt.axhline(y=95, linestyle="--", label="Clinical Threshold: Macro F1 = 95%")
plt.xlabel("Conv2d Sparsity (%)")
plt.ylabel("Performance (%)")
plt.title("EfficientNet-B0 Structured Pruning Trade-off")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(pruning_df["conv_sparsity"] * 100, pruning_df["compressed_size_mb"], marker="o")
plt.xlabel("Conv2d Sparsity (%)")
plt.ylabel("Compressed Size (MB)")
plt.title("Compressed Model Size vs Structured Pruning")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(pruning_df["conv_sparsity"] * 100, pruning_df["inference_time_ms_per_image"], marker="o")
plt.xlabel("Conv2d Sparsity (%)")
plt.ylabel("Inference Time (ms/image)")
plt.title("Inference Time vs Structured Pruning")
plt.grid(True)
plt.show()


## 13. Clinical viability summary

In [ ]:
clinical_threshold = 0.95
viable_models = pruning_df[pruning_df["test_macro_f1"] >= clinical_threshold]

print("Clinically Viable Models")
print("-" * 45)
display(viable_models)

if len(viable_models) > 0:
    best_compressed = viable_models.sort_values(by="conv_sparsity", ascending=False).iloc[0]

    print("\nMaximum Clinically Viable Compression")
    print("-" * 45)
    print(f"Model: {best_compressed['model']}")
    print(f"Target Pruning: {best_compressed['target_pruning'] * 100:.0f}%")
    print(f"Conv Sparsity: {best_compressed['conv_sparsity'] * 100:.2f}%")
    print(f"Total Sparsity: {best_compressed['total_sparsity'] * 100:.2f}%")
    print(f"Test Accuracy: {best_compressed['test_accuracy'] * 100:.2f}%")
    print(f"Test Macro F1: {best_compressed['test_macro_f1'] * 100:.2f}%")
    print(f"Compressed Size: {best_compressed['compressed_size_mb']:.2f} MB")
    print(f"Inference Time: {best_compressed['inference_time_ms_per_image']:.4f} ms/image")
else:
    print("No model met the clinical viability threshold of Macro F1 >= 0.95.")
